<a href="https://colab.research.google.com/github/tasfiah43/animaljamcalculator/blob/main/animal_jam_rarity_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Animal Jam Item Worth Wiki Scraper

Pulls item names + worth text from Animal Jam Worth Wiki

Uses free Mediawiki API

Script sleeps between requests + sets descriptive User-Agent

Installs:
  pip install requests
"""

import requests
import time
import csv
import re
import sys

In [ ]:
api_url = "https://aj-item-worth.fandom.com/api.php"
headers = {"User-Agent": "AJWorthDataProject/1.0 (personal ds project)"}
sleep_seconds = 0.5 # free API sleeping seconds

In [ ]:
def get_category_members(category_title, cmlimit=500):
  """Return a list of page titles in given wiki category"""
  members = []
  cmcontinue = None
  while True:
      params = {
        "actions": "query",
        "list": "categorymembers",
        "cmtitle": f"Category {category_title}",
        "cmlimit": cmlimit,
        "format": "json",
      }
      if cmcontinue:
        params["cmcontinue"] = cmcontinue

      resp = requests.get(api_url, params=params, headers=headers, timeout=15)
      resp.raise_for_status()
      data = resp.json()

      members += [m["title"] for m in data.get("query", {}).get("categorymembers", [])]

      cmcontinue = data.get("continue", {}).get("cmcontinue")

      if not cmcontinue:
        break
      time.sleep(sleep_seconds)

  return members

In [ ]:
def get_page_wikitext(title):
  """Return the raw wikitext of a page, or None if it fails"""

  params = {
      "actions": "parse",
      "page": title,
      "prop": "wikitext",
      "format": "json",
  }

  resp = requests.get(api_url, params=params, headers=headers, timeout=15)
  resp.raise_for_status()
  data = resp.json()

  if "error" in data:
    return None

  return data["parse"]["wikitext"]["*"]


In [ ]:
worth_field_patterns = [
    r"\|\s*worth\s*=\s*(.+)"
    r"\|\s*value\s*=\s*(.+)"
    r"\|\s*trade\s*value\s*=\s*(.+)"
]

def extract_worth(wikitext):
  """Pull a worth string out of page wikitext"""

  if not wikitext:
    return None

  for pattern in worth_field_patterns:
    match = re.search(pattern, wikitext, re.IGNORECASE)

    if match:
      return clean_wikitext(match.group(1))

  sentences = re.split(r"(?<=[.!?])\s+", wikitext)
  for s in sentences:
    if "worth" == s.lower():
      return clean_wikitext(s)

  return None

def clean_wikitext(text):
  """Strip common wiki markup"""
  text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)
  text = re.sub(r"\{\{.*?\}\}", "", text)
  text = re.sub(r"<ref.*?</ref>", "", text, flags=re.DOTALL)
  text = re.sub(r"'{2,}", "", text)
  return text.strip()

In [ ]:
def inspect_one_page(title):
  """Print raw wikitext for one page to see structure"""
  print(f"------Wikitext for '{title}' --------\n")
  print(wikitext)

In [ ]:
categories_to_scrape = [
    "Clothing Items",
    "Den Items",
    "Pets",
]

def scrape_all(output_csv="aj_item_worth.csv"):
  rows = []
  seen_titles = set()

  for category in categories_to_scrape:
    print(f"Fetching category: {category}")
    titles = get_category_members(category)
    print(f" found {len(titles)} pages")

    for title in titles:
      if title in seen_titles or title.startswith("Category:"):
        continue
      seen_titles.add(title)

      wikitext = get_page_wikitext(title)
      worth_text = extract_worth(wikitext)

      rows.append({
          "item_name": title,
          "category": category,
          "raw_worth_text": worth_text or "",
          "url": f"https://aj-item-worth.fandom.com/wiki/{title.replace(' ', '_')}",
      })

      time.sleep(sleep_seconds)

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
      writer = csv.DictWriter(f, fieldnames=["item_name", "category", "raw_worth_text", "url"])
      writer = writeheader()
      writer.writerows(rows)

    print(f"\nSaved {len(rows)} items to {output_csv}")

  if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "inspect":
      page_title = sys.argv[2] if len(sys.argv) > 2 else "Black Long"
      inspect_one_page(page_title)
    else:
      scrape_all()

In [ ]:
"""
Cleans scraped raw data
"""
import csv
from value_scale
import normalize_worth

input_file = "aj_item_worth.csv"
output_file = "aj_item_worth_clean.csv"

def main():
  rows = []
  parsed, unparsed = 0, 0

  with open(input_file, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
      score = normalize_worth(row["raw_worth_text"])
      row["normalized_worth"] = score if score is not None else ""
      if score is not None:
        parsed += 1
      else:
        unparsed += 1
      rows.append(row)

  if not rows:
    print("No rows found: did scraper.py run succesfully?")
    return

  fieldnames = list(rows[0].keys())
  with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

  total = parsed + unparsed
  print(f"Parsed {parsed}/{total} items automatically ({parsed/total:.0%}).")
  print(f"{unparsed} itrems need manual review or a better keyword/regex match.")
  print(f"Saved to {output_file}")

if __name__ == "__main__":
  main()

FileNotFoundError: [Errno 2] No such file or directory: 'aj_item_worth.csv'

In [ ]:
"""Value calculator"""

import collections import defaultDict

data_file = "aj_item_worth_clean.csv"

def load_data(path=datafile):
  lookup = defaultdict(list)
  try:
    with open(path, newline="", encoding="utf-8") as f:
      reader = csv.DictReader(f)
      for row in reader:
        score = row.get("normalized_worth", "")
        if score not in ("", "None"):
          lookup[row["item_name"].lower()].append(float(score))

  except FileNotFoundError:
    print(f"Couldn't find {path}. Run scraper.py and clean_data.py first.")
    sys.exit(1)

  return lookup

SyntaxError: invalid syntax (263322387.py, line 3)

In [ ]:
def average_worth(lookup, item_name):
  scores = lookup.get(item_name.lower())
  if not scores:
    return None
  return sum(scores) / len(scores)

def evaluate_side(lookup, item_name):
  total = 0
  unknown = []
  for name in item_names:
    avg = average_worth(lookup, name.strip())
    if avg is None:
      unknown.append(name.strip())
    else:
      total += avg
  return total, unknown

In [ ]:
def main():
  lookup = load_data()
  print("Animal Jam Trade Value Calculator")
  print("Type item names seperated by commas. Leave blank to skip a side.\n")

  side_a = [i for i in input("Your side (comma-seperated items):").split(",") if i.strip()]
  side_b = [i for i in input("Their side (comma-seperated items):").split(",") if i.strip()]

  total_a, unknown_a = evaluate_side(lookup, side_a)
  total_b, unknown_b = evaluate_side(lookup, side_b)

  print(f"\nYour side total score: {total_a:.if}")
  if unknown_a:
    print(f"  (not found {", ".join(unknown_a)})")

  print(f"\nYour side total score: {total_b:.if}")
  if unknown_b:
    print(f"  (not found {", ".join(unknown_b)})")

  diff = total_a - total_b
  print()
  if abs(diff) < 0.1:
    print("Roughly a fair trade")
  if diff > 0:
    print(f"Verdict: You're offering MORE value (by ~{diff:.1f}) points).")
  else:
    print(f"Verdict: You're offering LESS value (by ~{diff:.1f}) points).")

if __name__ == "__main__":
  main()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("aj_item_worth_clean.csv")
df["normalized_worth"] = pd.to_numeric(df["normalized_worth"], errors="coerce")

print("Rows:", len(df))
print("Rows with a parsed worth score:", df["normalized_worth"].notna().sum())

print("\nAverage worth by category:")
print(df.groupby("category")["normalized_worth"].mean().sort_values(ascending=False))

print("\nTop 10 most valuable items:")
print(
    df.sort_values("normalized_worth", ascending=False)[["item_name", "normalized_worth"]]
    .head(10)
    .to_string(index=False)
)

ax = df.groupby("category")["normalized_worth"].mean().sort_values().plot(
    kind="barh", figsize=(8, 5)
)

ax.set_xlabel("Average normalized worth score")
ax.set_title("Average Item Worth by Category")
plt.tight_layout()
plt.savefig("worth_by_category.png")
print("\nSaved chart to worth_by_category.png")

FileNotFoundError: [Errno 2] No such file or directory: 'aj_item_worth_clean.csv'